In [16]:
# Load dataset from local JSONL files (prepared by data.py)
import sys
sys.path.insert(0, '.')
from utils import load_metadata_rows, load_processor, build_dataset

# Load processor first
processor = load_processor()

# Load metadata from JSONL files (output from data.py)
train_rows = load_metadata_rows('datasets/train.jsonl', tokenizer=processor.tokenizer)
val_rows = load_metadata_rows('datasets/val.jsonl', tokenizer=processor.tokenizer)
test_rows = load_metadata_rows('datasets/test.jsonl', tokenizer=processor.tokenizer)

# Build datasets
train_dataset = build_dataset(train_rows, processor)
val_dataset = build_dataset(val_rows, processor)
test_dataset = build_dataset(test_rows, processor)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')
train_dataset[0]['text']

"AS THEY'RE LEAVING <COMMA> CAN KASH PULL ZAHRA ASIDE REALLY QUICKLY <QUESTIONMARK>"

In [17]:
# Load model from local (like train.py)
import torch
from utils import load_model_and_processor

model, processor = load_model_and_processor()

# Move to device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'Model loaded on {device}')

Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 14.22it/s]


In [22]:
import evaluate
from whisper.normalizers import EnglishTextNormalizer
from transformers.feature_extraction_utils import BatchFeature
from torch.utils.data import DataLoader
import tqdm


def extract_audio_array(audio):
    """Handle both legacy dict format and new torchcodec AudioDecoder objects."""
    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        return samples.data.squeeze(0).numpy()
    elif isinstance(audio, dict):
        return audio["array"]
    return audio


class GraniteCollator:
    def __init__(self, processor, inference_mode=False):
        self.processor = processor
        self.inference_mode = inference_mode

    def __call__(self, examples):
        prompts = [example["prompt"] for example in examples]
        audios = [extract_audio_array(example["audio"]) for example in examples]

        processed = self.processor(prompts, audios, return_tensors="pt", padding=True, padding_side="left")
        input_ids = processed.input_ids
        attention_mask = processed.attention_mask
        labels = None
        # tokenize targets
        if not self.inference_mode:
            targets = [example["text"] + self.processor.tokenizer.eos_token for example in examples]
            targets = self.processor.tokenizer(targets, return_tensors="pt", padding=True, padding_side="right")
            # combine prompt+targets
            input_ids = torch.cat([input_ids, targets.input_ids], dim=1)
            attention_mask = torch.cat([attention_mask, targets.attention_mask], dim=1)
            labels = targets.input_ids.clone()
            # Set non-target tokens to -100 for loss calculation
            labels[~(targets.attention_mask.bool())] = -100  
            labels = torch.cat([torch.full_like(processed.input_ids, -100), labels], dim=1)

        return BatchFeature(data={
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "input_features": processed.input_features,
            "input_features_mask": processed.input_features_mask
        })

def compute_wer(model, processor, cur_dataset):
    collator = GraniteCollator(processor, inference_mode=True)
    dataloader = DataLoader(cur_dataset, batch_size=16, collate_fn=collator, num_workers=8)
    normalizer = EnglishTextNormalizer()
    wer_metric = evaluate.load("wer")
    model = model.eval().cuda()
    
    all_outputs = []
    for batch in tqdm.tqdm(dataloader, desc="Running inference"):
        batch = batch.to("cuda")
        with torch.inference_mode(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model.generate(**batch, max_new_tokens=400, num_beams=4, early_stopping=True)
        input_length = batch.input_ids.shape[1]
        outputs = outputs[:, input_length:].cpu()
        for x in outputs:
            all_outputs.append(processor.tokenizer.decode(x, skip_special_tokens=True))
        
    gt_texts = [normalizer(x) for x in cur_dataset["text"]]
    all_outputs = [normalizer(x) for x in all_outputs]
    wer = wer_metric.compute(references=gt_texts, predictions=all_outputs)
    return wer


In [24]:
from transformers import TrainingArguments, Trainer

for n, p in model.named_parameters():
    # tranining only the projector/lora layers
    p.requires_grad = "projector" in n or "lora" in n

args = TrainingArguments(
    output_dir="save_dir",
    remove_unused_columns=False,
    report_to="none",
    bf16=True,
    eval_strategy="steps",
    save_strategy="no",
    eval_steps=0.1,
    dataloader_num_workers=16,
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16, 
    gradient_accumulation_steps=2,
    num_train_epochs=1.0,
    warmup_ratio=0.2,
    logging_steps=0.1,
    learning_rate=3e-5,
    data_seed=42,
)
data_collator = GraniteCollator(processor)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=processor,
)
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 100257, 'bos_token_id': 100257, 'pad_token_id': 100256}.


Step,Training Loss,Validation Loss
16,1.821600,0.832424
32,1.623000,0.781848
48,1.353800,0.724028
64,1.093500,0.681368
80,0.963500,0.654560
96,0.936400,0.636325
112,0.882600,0.629478
128,0.878200,0.622325
144,0.909000,0.624130


TrainOutput(global_step=157, training_loss=1.131993336282718, metrics={'train_runtime': 105.8504, 'train_samples_per_second': 47.236, 'train_steps_per_second': 1.483, 'total_flos': 9837761710902336.0, 'train_loss': 1.131993336282718, 'epoch': 1.0})

In [25]:
wer_after_train = compute_wer(model, processor, test_dataset)

print(f"WER after finetuning {wer_after_train*100:.3f}")
print(f"WER improvement {(wer_before_train - wer_after_train)*100:.3f}")

Running inference: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:21<00:00,  2.15s/it]

WER after finetuning 9.419
WER improvement 0.367
